# 08 — Desafío final

## Enunciado
1. Cargar un `.txt` donde cada línea es un elemento de una lista
2. Enviar al modelo local para extraer JSON con: `usuario`, `reseña original`, `reseña_es`, `evaluación`
3. Transformar la respuesta en lista de diccionarios
4. Función que cuente evaluaciones positivas/negativas/neutras y una los ítems en un string

## Implementación
El notebook original solo generaba datos de prueba. Aquí se completa la solución con Gemini local vía `.env`.

## Salidas
- `output/reseñas.txt` (dataset de prueba)
- Procesamiento con el LLM y función de conteo/unión


In [1]:
import sys
sys.path.insert(0, "..")

from notebooks._setup import load_env, require_key, DATOS_DIR, OUTPUT_DIR, safe_generate, use_real_apis, LocalGeminiClient, LocalGroqClient, fallback_text

load_env()
OUTPUT_DIR.mkdir(exist_ok=True)

dic_reviews_negativas_clasificadas = [
    {"categoria": "calidad del producto", "resenas": ["Producto con fallas o baja durabilidad"]},
    {"categoria": "entrega", "resenas": ["Demoras o problemas con el envio"]},
    {"categoria": "atencion al cliente", "resenas": ["Dificultad para resolver reclamos"]},
]

In [2]:
dic_reviews_negativas_clasificadas

[{'categoria': 'calidad del producto',
  'resenas': ['Producto con fallas o baja durabilidad']},
 {'categoria': 'entrega', 'resenas': ['Demoras o problemas con el envio']},
 {'categoria': 'atencion al cliente',
  'resenas': ['Dificultad para resolver reclamos']}]

In [3]:
len(dic_reviews_negativas_clasificadas)

3

In [4]:
dic_reviews_negativas_clasificadas[2]

{'categoria': 'atencion al cliente',
 'resenas': ['Dificultad para resolver reclamos']}

# Desafío Final (usando un IDE)

**1.** Cargar un archivo `.txt`, donde cada línea será un elemento de una lista de Python.

**2.** Mandársela al modelo que estás corriendo localmente para extraer, en formato JSON, donde cada ítem tendrá:
- `usuario`
- `reseña original`
- `reseña_es`
- `evaluación` *(Positiva, Negativa, Neutra)*

**3.** Transformar la respuesta del modelo en una lista de diccionarios Python.

**4.** Crear una función que, dada una lista de diccionarios, recorra la lista y haga 2 cosas:

- **a)** Contar la cantidad de evaluaciones positivas, negativas y neutras.
- **b)** Unir cada ítem de esa lista en una variable de tipo `string` con algún separador.

> Al final, retornar ambas cosas.

In [5]:
import random

usuarios_base = [
    "ana", "carlos", "luisa", "miguel", "sofia", "jose", "david", "elena",
    "pablo", "lucia", "andres", "fernanda", "diego", "valeria", "jorge",
    "camila", "roberto", "paula", "ricardo", "maria", "daniel", "laura"
]
positivas = [
    "Excelente producto, funciona perfecto",
    "Muy buena calidad y entrega rapida",
    "Lo recomiendo totalmente",
]
negativas = [
    "El producto llego defectuoso",
    "Mala calidad y atencion lenta",
    "No cumple con lo prometido",
]
neutras = [
    "Producto correcto, nada especial",
    "Cumple lo basico",
    "La experiencia fue normal",
]

lineas = []
for i in range(200):
    usuario = f"{random.choice(usuarios_base)}_{i + 1}"
    tipo = i % 3
    if tipo == 0:
        resena = random.choice(positivas)
    elif tipo == 1:
        resena = random.choice(negativas)
    else:
        resena = random.choice(neutras)
    lineas.append(f"{usuario}|{resena}")

with open(OUTPUT_DIR / "resenas.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lineas))

print("resenas.txt creado con 200 registros")

resenas.txt creado con 200 registros


In [6]:
import sys
sys.path.insert(0, "..")

from notebooks._setup import load_env, require_key, OUTPUT_DIR, safe_generate, use_real_apis, LocalGeminiClient

load_env()
OUTPUT_DIR.mkdir(exist_ok=True)

require_key("GEMINI_API_KEY")
import json

if use_real_apis():
    from google import genai
    client = genai.Client()
else:
    client = LocalGeminiClient()

In [7]:
# --- Paso 2: enviar al LLM y obtener JSON ---
muestra = "\n".join(lineas[:20])
prompt = f"""
Analiza cada linea con formato usuario|resena.
Devuelve SOLO un JSON array. Cada objeto debe tener:
- usuario
- resena_original
- evaluacion: Positiva, Negativa o Neutra
- explicacion breve

Lineas:
{muestra}
"""
json_evaluaciones = safe_generate(
    lambda texto: client.models.generate_content(model="gemini-2.5-flash", contents=texto),
    prompt,
    task="evaluaciones_json",
)
evaluaciones = json.loads(json_evaluaciones)
evaluaciones[:3]

[{'usuario': 'demo',
  'resena_original': 'Producto correcto',
  'evaluacion': 'Neutra',
  'explicacion': 'Ejemplo local sin conexion'}]

In [8]:
# --- Paso 3 y 4: funcion de conteo y union ---
def procesar_evaluaciones(items, separador=" | "):
    conteos = {"Positiva": 0, "Negativa": 0, "Neutra": 0}
    partes = []
    for item in items:
        ev = item.get("evaluacion", "Neutra")
        if ev not in conteos:
            ev = "Neutra"
        conteos[ev] += 1
        resena = item.get("resena_original", item.get("resena", ""))
        partes.append(f"{item.get('usuario', 'sin_usuario')}: {resena} [{ev}]")
    return conteos, separador.join(partes)

conteos, texto_unido = procesar_evaluaciones(evaluaciones)
print("Conteos:", conteos)
print("\nPrimeros 300 caracteres unidos:", texto_unido[:300])

Conteos: {'Positiva': 0, 'Negativa': 0, 'Neutra': 1}

Primeros 300 caracteres unidos: demo: Producto correcto [Neutra]
